Add influxdb credentials fine to your environment. I suggest you do create a file 
```
mkdir /home/<USERNAME>/credentials
```
Add this file to your PATH: edit your .bashrc or .bash_profile and add the following line:
```
export CREDENTIALS="/home/<USERNAME>/credentials"
```
Then create a file `/home/<USERNAME>/credentials/influxdb.yaml` with the following content:
```
influxdb:
  url: <ADD URL>
  token: <ADD TOKEN>
  org: 'FDL'
  bucket: 'hl-therm-influxdb'
```

Make sure you have installed karman:
```
pip install -e .
```

In [ ]:
from karman.io import InfluxDBManager
from datetime import datetime
import pandas as pd

In [ ]:
# connect to the database
db = InfluxDBManager()

In [ ]:
# Check that you can connect to the database
print(db.client.ping())
print("")
print(db.health())


In [ ]:
# check what buckets are available
print(db.get_bucket_names())

In [ ]:
start = "2010-01-01T00:00:00Z"
stop = "2010-02-02T00:00:00Z"

# What data you want to get out of the db
columns = ['tudelft_thermo__altitude__[m]',
       'tudelft_thermo__longitude__[deg]', 'tudelft_thermo__latitude__[deg]',
       'tudelft_thermo__ground_truth_thermospheric_density__[kg/m**3]',
       'tudelft_thermo__satellite__', 'all__year__[y]',
       'all__day_of_year__[d]', 'all__seconds_in_day__[s]',
       'space_environment_technologies__f107_obs__',
       'space_environment_technologies__f107_average__',
       'space_environment_technologies__s107_obs__',
       'space_environment_technologies__s107_average__',
       'space_environment_technologies__m107_obs__',
       'space_environment_technologies__m107_average__',
       'space_environment_technologies__y107_obs__',
       'space_environment_technologies__y107_average__', 'JB08__d_st_dt__[K]',
       'celestrack__ap_average__']

# Read a segment of informaton from the database
df = db.query_single_table_daterange(start=start, stop=stop, columns=columns)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df["tudelft_thermo__satellite__"].unique()

In [ ]:
df_in = pd.read_parquet("/shared/temp/tudelft_version_01_GOCE_data_db_GO_DNS_WND_ACC_2009_11_v01.parquet")
df_in.head()

In [ ]:
df_in = df_in.set_index("all__dates_datetime__")

db.write(df_in.head(100), measurement="satellites_data_w_sw", field_columns=df_in.columns)

In [ ]:
start = "2000-01-01T00:00:00Z"
end = "2000-12-31T23:59:59Z"
db._delete_everything_in_range(start=start, stop=end)

In [ ]:
# for year in range(2000, 2024+1):
#     start = f"{year}-01-01T00:00:00Z"
#     end = f"{year}-12-31T23:59:59Z"
#     db._delete_everything_in_range(start=start, stop=end)